In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.exists('/content/work/data/vessel_catalog.csv'):
    os.system('mkdir -p /content/work/data /content/work/model /content/work/output /content/raw /content/baseline')
    os.system('unzip -o -q /content/drive/MyDrive/vessel/open_data.zip -d /content/raw')
    os.system('unzip -o -q /content/raw/open.zip -d /content/work/data')
    os.system('unzip -o -q /content/raw/baseline_submit.zip -d /content/baseline')
print(os.listdir('/content/work/data'))

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

D = '/content/work/data'
train = pd.read_csv(f'{D}/train.csv')
cat   = pd.read_csv(f'{D}/vessel_catalog.csv')
sub   = pd.read_csv(f'{D}/sample_submission.csv')

print(train.shape, cat.shape, sub.shape)

In [ ]:
print('== train 컬럼')
print(train.dtypes)
print('\n== catalog 컬럼')
print(cat.dtypes)

display(train.head(10))
display(cat.head(10))

In [ ]:
m = train.merge(cat, left_on='target_vessel_id', right_on='vessel_id', suffixes=('_rec', '_cat'))

cols = ['vessel_name_rec', 'vessel_name_cat',
        'gross_tonnage_rec', 'gross_tonnage_cat',
        'flag_name_rec', 'flag_name_cat',
        'vessel_type_rec', 'vessel_type_cat']
print(m[cols].sample(20, random_state=0).to_string())

In [ ]:
import re

# 한글 선박명 기록 비율
m['is_kor'] = m['vessel_name_rec'].astype(str).str.contains('[가-힣]')
print('한글 선박명 비율:', round(m['is_kor'].mean(), 3))

# 총톤수: 결측 비율, 정확히 일치하는 비율, 상대 차이 분포
gt_rec, gt_cat = m['gross_tonnage_rec'], m['gross_tonnage_cat']
print('기록 톤수 결측 비율:', round(gt_rec.isna().mean(), 3))
both = gt_rec.notna() & gt_cat.notna()
print('톤수 정확히 일치:', round((gt_rec[both] == gt_cat[both]).mean(), 3))
rel = (gt_rec[both] - gt_cat[both]).abs() / gt_cat[both]
print('톤수 상대 차이 분포:')
print(rel.describe(percentiles=[.5, .75, .9, .95, .99]).round(3))

In [ ]:
!ls -laR /content/baseline

In [ ]:
!cat /content/baseline/script.py
!echo '-----'
!cat /content/baseline/requirements.txt

In [ ]:
cs = train['call_sign']

# 1) 호출부호가 비어 있는 비율
print('호출부호 결측 비율:', round(cs.isna().mean(), 3))

# 2) 같은 호출부호가 train에 여러 번 나오는지
counts = cs.value_counts()
print('고유 호출부호 수:', len(counts), '/ 전체 행:', len(train))
print('2번 이상 나오는 호출부호 비율:', round((counts >= 2).mean(), 3))

# 3) 호출부호가 같으면 정답도 항상 같은지
n_ans = train.groupby('call_sign')['target_vessel_id'].nunique()
print('호출부호 하나에 정답이 1개뿐인 비율:', round((n_ans == 1).mean(), 3))

In [ ]:
from sklearn.model_selection import train_test_split

tr, va = train_test_split(train, test_size=0.2, random_state=42)

# 80% 쪽으로 호출부호 → 정답 사전 만들기
cs2id = dict(zip(tr['call_sign'], tr['target_vessel_id']))

# 20% 쪽 기록 중 호출부호가 사전에 있는 비율, 그리고 그걸로 맞힌 비율
found = va['call_sign'].isin(cs2id)
pred = va['call_sign'].map(cs2id)
print('호출부호로 찾을 수 있는 비율:', round(found.mean(), 3))
print('찾은 것 중 정답 비율:', round((pred[found] == va.loc[found, 'target_vessel_id']).mean(), 3))

In [ ]:
!rm -rf /content/submit_v2 && cp -r /content/baseline /content/submit_v2

lookup = train[['call_sign', 'target_vessel_id']].drop_duplicates('call_sign')
lookup.to_csv('/content/submit_v2/model/cs2id.csv', index=False, encoding='utf-8')
print('대응표 행 수:', len(lookup))
!ls -la /content/submit_v2/model

In [ ]:
%%writefile /content/submit_v2/script.py
"""Baseline RandomForest + call_sign lookup built from train.csv."""

from pathlib import Path
import sys

import joblib
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent / "model"))
from baseline_lib import CatalogIndex, FEATURE_NAMES, predict_ids


def run(
    test_path="data/test.csv",
    catalog_path="data/vessel_catalog.csv",
    model_path="model/rf.pkl",
    lookup_path="model/cs2id.csv",
    output_path="output/submission.csv",
):
    test = pd.read_csv(test_path, encoding="utf-8-sig")
    catalog = pd.read_csv(catalog_path, encoding="utf-8-sig")
    bundle = joblib.load(model_path)
    if tuple(bundle["feature_names"]) != FEATURE_NAMES:
        raise ValueError("Model and feature code versions do not match")
    if test["sample_id"].isna().any() or not test["sample_id"].is_unique:
        raise ValueError("test.csv sample_id must be non-null and unique")

    index = CatalogIndex(catalog)

    # 1) 베이스라인 예측
    base_pred = pd.Series(list(predict_ids(bundle["model"], test, index)), index=test.index)

    # 2) 호출부호 대응표로 찾을 수 있는 행은 덮어쓰기
    lookup = pd.read_csv(lookup_path, encoding="utf-8")
    cs2id = {
        cs: vid
        for cs, vid in zip(lookup["call_sign"], lookup["target_vessel_id"])
        if vid in index.id_to_index
    }
    cs_pred = test["call_sign"].map(cs2id)
    predictions = cs_pred.fillna(base_pred).tolist()

    if len(predictions) != len(test) or not set(predictions).issubset(index.id_to_index):
        raise ValueError("Invalid catalog predictions")
    result = pd.DataFrame({"sample_id": test["sample_id"], "target_vessel_id": predictions})
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Saved {len(result)} predictions to {output_path}")
    return output_path


if __name__ == "__main__":
    run()

In [ ]:
!rm -rf /content/test_run && cp -r /content/submit_v2 /content/test_run
!mkdir -p /content/test_run/data
!cp /content/work/data/vessel_catalog.csv /content/test_run/data/

# 가짜 test.csv: train 일부에서 정답 컬럼만 뺀 것
train.sample(500, random_state=0).drop(columns=['target_vessel_id']) \
     .to_csv('/content/test_run/data/test.csv', index=False)

!cd /content/test_run && python script.py
!head -5 /content/test_run/output/submission.csv

In [ ]:
!cd /content/submit_v2 && rm -f /content/submit.zip && zip -r /content/submit.zip script.py requirements.txt model
!unzip -l /content/submit.zip

from google.colab import files
files.download('/content/submit.zip')

In [ ]:
import pandas as pd

# 2번 확인: 로컬 테스트 결과가 정답과 얼마나 맞는지
out = pd.read_csv('/content/test_run/output/submission.csv')
truth = train.set_index('sample_id').loc[out['sample_id'], 'target_vessel_id'].values
print('로컬 테스트 정확도:', round((out['target_vessel_id'].values == truth).mean(), 3))

# 1번 확인: 제출한 zip 안에 대응표가 들어 있는지
!unzip -l /content/submit.zip